In [1]:
import pandas as pd
import re

In [2]:
# Load College Scorecard Field of Study data
scorecard_fos = pd.read_csv(r"C:\capstone_data\raw\Most-Recent-Cohorts-Field-of-Study.csv")

# Load College Scorecard Institution Level data
scorecard_il = pd.read_csv(r"C:\capstone_data\raw\Most-Recent-Cohorts-Institution.csv")

C:\Users\sigep\AppData\Local\Temp\ipykernel_60260\2175691476.py:5: DtypeWarning: Columns (9,1407,1408,1431,1432,1532,1537,1538,1539,1540,1542,1546,1589,1601,1602,1606,1608,1611,1614,1615,1616,1619,1620,1621,1622,1623,1624,1625,1626,1627,1628,1629,1653,1679,1690,1692,1697,1700,1702,1725,1726,1727,1728,1729,1743,1815,1816,1817,1818,1823,1824,1830,1831,1879,1880,1881,1882,1883,1884,1885,1886,1887,1888,1889,1890,1891,1892,1893,1894,1895,1896,1897,1898,1909,1910,1911,1912,1913,1957,1958,1959,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1983,1984,2376,2377,2403,2404,2495,2496,2497,2498,2499,2500,2501,2502,2503,2504,2505,2506,2507,2508,2509,2510,2511,2512,2513,2514,2515,2516,2517,2518,2519,2520,2521,2522,2523,2524,2525,2526,2527,2528,2529,2530,2958,3215,3231,3235,3236) have mixed types. Specify dtype option on import or set low_memory=False.
  scorecard_il = pd.read_csv(r"C:\capstone_data\raw\Most-Recent-Cohorts-Institution.csv")


In [3]:
scorecard_fos.shape

(229188, 174)

In [4]:
scorecard_fos.columns

Index(['UNITID', 'OPEID6', 'INSTNM', 'CONTROL', 'MAIN', 'CIPCODE', 'CIPDESC',
       'CREDLEV', 'CREDDESC', 'IPEDSCOUNT1',
       ...
       'EARN_COUNT_PELL_WNE_5YR', 'EARN_PELL_WNE_MDN_5YR',
       'EARN_COUNT_NOPELL_WNE_5YR', 'EARN_NOPELL_WNE_MDN_5YR',
       'EARN_COUNT_MALE_WNE_5YR', 'EARN_MALE_WNE_MDN_5YR',
       'EARN_COUNT_NOMALE_WNE_5YR', 'EARN_NOMALE_WNE_MDN_5YR',
       'EARN_COUNT_HIGH_CRED_5YR', 'EARN_IN_STATE_5YR'],
      dtype='object', length=174)

In [5]:
# Find earnings columns that may be useful later
[col for col in scorecard_fos.columns if "MDN_3YR".lower() in col.lower()]

['EARN_NE_MDN_3YR',
 'EARN_PELL_NE_MDN_3YR',
 'EARN_NOPELL_NE_MDN_3YR',
 'EARN_MALE_NE_MDN_3YR',
 'EARN_NOMALE_NE_MDN_3YR']

In [6]:
# Select columns from scorecard to keep. EARN_NE_MDN_3YR provides median earnings 3 years after completion for non-enrolled students.
scorecard_fos_clean = scorecard_fos[[
    "UNITID"
    ,"INSTNM"
    ,"CIPCODE"
    ,"CIPDESC"
    ,"CREDLEV"
    ,"CREDDESC"
    ,"EARN_MDN_1YR"
    ,"EARN_NE_MDN_3YR"
    ,"EARN_MDN_5YR"
]].copy()

In [7]:
scorecard_fos_clean.rename(columns={
    "UNITID" : "unitid"
    ,"INSTNM" : "inst_name"
    ,"CIPCODE" : "cipcode"
    ,"CIPDESC" : "cip_desc"
    ,"CREDLEV" : "credlev"
    ,"CREDDESC" : "cred_desc"
    ,"EARN_MDN_1YR" : "median_earnings_1yr"
    ,"EARN_NE_MDN_3YR" : "median_earnings_3yr"
    ,"EARN_MDN_5YR" : "median_earnings_5yr"
}, inplace = True)

In [8]:
# Fix cip code to uniform xxxx dtype str format.
# Fill three digit codes to 4 digit with leading 0
scorecard_fos_clean["cip4"]= (
    scorecard_fos_clean["cipcode"]
    .astype(str)
    .str.zfill(4)
)

In [9]:
#Explore scorecard_il
scorecard_il.shape

(6429, 3306)

In [10]:
# Select columns from scorecard to keep.
scorecard_il_clean = scorecard_il[[
    "UNITID"
    ,"STABBR"
    ,"CONTROL"
    ,"COSTT4_A"
    ,"NPT4_PUB"
    ,"NPT4_PRIV"
]].copy()

In [11]:
# Clean columns for naming conventions
scorecard_il_clean.rename(columns={
    "UNITID" : "unitid"
    ,"STABBR" : "state"
    ,"CONTROL" : "control"
    ,"COSTT4_A" : "avg_annual_cost"
    ,"NPT4_PUB" : "avg_net_price_pub"
    ,"NPT4_PRIV" : "avg_net_price_priv"
}, inplace = True)

In [12]:
scorecard_il_clean.columns

Index(['unitid', 'state', 'control', 'avg_annual_cost', 'avg_net_price_pub',
       'avg_net_price_priv'],
      dtype='object')

In [13]:
# Get unitids for Tennessee institutions
tn_unitids = scorecard_il_clean[scorecard_il_clean["state"] == "TN"]["unitid"]

# Filter the field of study dataframe to only include Tennessee institutions
scorecard_fos_tn = scorecard_fos_clean[scorecard_fos_clean["unitid"].isin(tn_unitids)]

# Filter the institution-level dataframe to only include Tennessee institutions
scorecard_il_tn = scorecard_il_clean[scorecard_il_clean["state"] == "TN"]

In [14]:
scorecard_fos_tn.info

<bound method DataFrame.info of           unitid                                 inst_name  cipcode  \
162267  219505.0                  American Baptist College     2401   
162268  219505.0                  American Baptist College     2401   
162269  219505.0                  American Baptist College     2401   
162270  219505.0                  American Baptist College     3802   
162271  219505.0                  American Baptist College     3906   
...          ...                                       ...      ...   
219044  497125.0  Galen Health Institutes-Nashville Campus     5138   
219048  497198.0          Academy of Allied Health Careers     5107   
219049  497198.0          Academy of Allied Health Careers     5108   
219050  497198.0          Academy of Allied Health Careers     5110   
219051  497198.0          Academy of Allied Health Careers     5139   

                                                 cip_desc  credlev  \
162267  Liberal Arts and Sciences, General St

In [15]:
scorecard_il_tn.info

<bound method DataFrame.info of         unitid state  control  avg_annual_cost  avg_net_price_pub  \
2961    219505    TN        2          27241.0                NaN   
2962    219587    TN        3              NaN                NaN   
2963    219596    TN        1              NaN             3720.0   
2964    219602    TN        1          26790.0            14846.0   
2965    219639    TN        2          21438.0                NaN   
...        ...   ...      ...              ...                ...   
6300  44376603    TN        3              NaN                NaN   
6346  45519610    TN        3              NaN                NaN   
6350  45519614    TN        3              NaN                NaN   
6373  47503304    TN        2              NaN                NaN   
6379  47503310    TN        2              NaN                NaN   

      avg_net_price_priv  
2961             19294.0  
2962             10343.0  
2963                 NaN  
2964                 NaN  
2965

In [16]:
# Is NSS in the data set?
scorecard_fos_tn[scorecard_fos_tn["inst_name"].str.contains("Nashville Software", case=False, na=False)]

,unitid,inst_name,cipcode,cip_desc,credlev,cred_desc,median_earnings_1yr,median_earnings_3yr,median_earnings_5yr,cip4


In [17]:
scorecard_fos_tn["cipcode"].value_counts()

cipcode
5202    134
5138    113
2401     85
5109     73
1312     68
       ... 
405       1
406       1
105       1
1106      1
5126      1
Name: count, Length: 292, dtype: int64

In [18]:
scorecard_fos_tn["cipcode"].unique()

array([2401, 3802, 3906, 3999, 4400, 5009, 5207, 1204, 1504, 4603, 4701,
       4703, 4706, 4805, 5108, 5139, 5204,  100,  901,  904, 1101, 1105,
       1107, 1108, 1301, 1303, 1304, 1306, 1310, 1311, 1312, 1313, 1314,
       1412, 1500, 1506, 1601, 1612, 2301, 2601, 2611, 2701, 3105, 3801,
       4005, 4006, 4008, 4201, 4228, 4301, 4407, 4510, 4511, 4901, 5005,
       5007, 5107, 5109, 5110, 5111, 5115, 5138, 5201, 5202, 5203, 5208,
       5214, 5401, 5122,  301,  402,  408,  501,  909,  910, 1002, 1399,
       1605, 1609, 1611, 2200, 2201, 2602, 2615, 2703, 3099, 3800, 3902,
       3905, 4506, 4509, 5004, 5006, 5010, 5120, 5123, 5206, 5211, 5212,
       5213, 5219, 2613, 3005, 4299, 4303, 5100, 1401, 2313, 3001, 3008,
       3010, 3071, 3904, 5299, 1499, 1901, 1902, 1904, 1905, 1906, 1907,
       1909, 2699, 3101, 3899, 4402, 4599, 5131, 5132,  106,  183, 1503,
       1507, 1508, 1513, 2203, 3000, 3201, 4103, 4302, 4702, 4902, 5106,
       5135, 5199, 5209,  502, 1110, 1407, 1408, 14

In [19]:
scorecard_fos_tn.dtypes

unitid                 float64
inst_name               object
cipcode                  int64
cip_desc                object
credlev                  int64
cred_desc               object
median_earnings_1yr     object
median_earnings_3yr     object
median_earnings_5yr     object
cip4                    object
dtype: object

In [20]:
scorecard_il_tn.dtypes

unitid                  int64
state                  object
control                 int64
avg_annual_cost       float64
avg_net_price_pub     float64
avg_net_price_priv    float64
dtype: object

In [21]:
# Join scorecard data sets
scorecard_tn = scorecard_fos_tn.merge(
    scorecard_il_tn,
    on='unitid',
    how='left'
)

In [22]:
scorecard_tn[['credlev', 'cred_desc']].head()

,credlev,cred_desc
0,2,Associate's Degree
1,3,Bachelor's Degree
2,4,Post-baccalaureate Certificate
3,1,Undergraduate Certificate or Diploma
4,1,Undergraduate Certificate or Diploma


In [23]:
scorecard_tn[['median_earnings_1yr',
               'median_earnings_3yr',
               'median_earnings_5yr']].describe()

,median_earnings_1yr,median_earnings_3yr,median_earnings_5yr
count,4166,3681,4166
unique,1083,878,1022
top,PS,PS,PS
freq,3067,2787,3125


In [24]:
scorecard_tn.groupby('credlev')[
    ['median_earnings_1yr',
     'median_earnings_3yr',
     'median_earnings_5yr']
].apply(lambda x: x.isnull().mean())

,median_earnings_1yr,median_earnings_3yr,median_earnings_5yr
credlev,,,
1,0.0,0.165254,0.0
2,0.0,0.157692,0.0
3,0.0,0.072997,0.0
4,0.0,0.533333,0.0
5,0.0,0.118384,0.0
6,0.0,0.076577,0.0
7,0.0,0.193548,0.0
8,0.0,0.173729,0.0


In [25]:
scorecard_tn['cred_desc'].value_counts()

cred_desc
Bachelor's Degree                       1685
Master's Degree                          718
Undergraduate Certificate or Diploma     708
Associate's Degree                       520
Graduate/Professional Certificate        236
Doctoral Degree                          222
First Professional Degree                 62
Post-baccalaureate Certificate            15
Name: count, dtype: int64

In [26]:
scorecard_tn.groupby(['cip4', 'cred_desc']).size().sort_values(ascending=False)

cip4  cred_desc                           
1204  Undergraduate Certificate or Diploma    47
5202  Bachelor's Degree                       38
2601  Bachelor's Degree                       37
5401  Bachelor's Degree                       36
2301  Bachelor's Degree                       36
                                              ..
0111  Master's Degree                          1
0112  Bachelor's Degree                        1
      Master's Degree                          1
0180  Doctoral Degree                          1
0181  Master's Degree                          1
Length: 855, dtype: int64

In [27]:
scorecard_tn['cred_desc'].value_counts()

cred_desc
Bachelor's Degree                       1685
Master's Degree                          718
Undergraduate Certificate or Diploma     708
Associate's Degree                       520
Graduate/Professional Certificate        236
Doctoral Degree                          222
First Professional Degree                 62
Post-baccalaureate Certificate            15
Name: count, dtype: int64

In [28]:
scorecard_tn.head()

,unitid,inst_name,cipcode,cip_desc,credlev,cred_desc,median_earnings_1yr,median_earnings_3yr,median_earnings_5yr,cip4,state,control,avg_annual_cost,avg_net_price_pub,avg_net_price_priv
0,219505.0,American Baptist College,2401,"Liberal Arts and Sciences, General Studies and...",2,Associate's Degree,PS,PS,PS,2401,TN,2,27241.0,NaN,19294.0
1,219505.0,American Baptist College,2401,"Liberal Arts and Sciences, General Studies and...",3,Bachelor's Degree,PS,PS,PS,2401,TN,2,27241.0,NaN,19294.0
2,219505.0,American Baptist College,2401,"Liberal Arts and Sciences, General Studies and...",4,Post-baccalaureate Certificate,PS,PS,PS,2401,TN,2,27241.0,NaN,19294.0
3,219505.0,American Baptist College,3802,Religion/Religious Studies.,1,Undergraduate Certificate or Diploma,PS,PS,PS,3802,TN,2,27241.0,NaN,19294.0
4,219505.0,American Baptist College,3906,Theological and Ministerial Studies.,1,Undergraduate Certificate or Diploma,PS,PS,PS,3906,TN,2,27241.0,NaN,19294.0


In [29]:
scorecard_tn.tail()

,unitid,inst_name,cipcode,cip_desc,credlev,cred_desc,median_earnings_1yr,median_earnings_3yr,median_earnings_5yr,cip4,state,control,avg_annual_cost,avg_net_price_pub,avg_net_price_priv
4161,497125.0,Galen Health Institutes-Nashville Campus,5138,"Registered Nursing, Nursing Administration, Nu...",3,Bachelor's Degree,76234,72065,89101,5138,TN,3,31743.0,NaN,27306.0
4162,497198.0,Academy of Allied Health Careers,5107,Health and Medical Administrative Services.,1,Undergraduate Certificate or Diploma,PS,NaN,PS,5107,TN,3,NaN,NaN,9095.0
4163,497198.0,Academy of Allied Health Careers,5108,Allied Health and Medical Assisting Services.,1,Undergraduate Certificate or Diploma,PS,NaN,PS,5108,TN,3,NaN,NaN,9095.0
4164,497198.0,Academy of Allied Health Careers,5110,Clinical/Medical Laboratory Science/Research a...,1,Undergraduate Certificate or Diploma,PS,NaN,PS,5110,TN,3,NaN,NaN,9095.0
4165,497198.0,Academy of Allied Health Careers,5139,"Practical Nursing, Vocational Nursing and Nurs...",1,Undergraduate Certificate or Diploma,PS,NaN,PS,5139,TN,3,NaN,NaN,9095.0


In [30]:
# Insepect earnings for viability
earn_1yr = scorecard_tn['median_earnings_1yr'].value_counts().to_frame()
earn_1yr

,count
median_earnings_1yr,
PS,3067
26952,3
84311,2
52528,2
52331,2
...,...
22826,1
24411,1
31401,1


In [31]:
earn_3yr = scorecard_tn['median_earnings_3yr'].value_counts().to_frame()
earn_3yr

,count
median_earnings_3yr,
PS,2787
39833,2
36609,2
33693,2
51761,2
...,...
25930,1
36908,1
39004,1


In [32]:
earn_5yr = scorecard_tn['median_earnings_5yr'].value_counts().to_frame()
earn_5yr

,count
median_earnings_5yr,
PS,3125
47664,2
44034,2
40094,2
28474,2
...,...
28737,1
50175,1
43920,1


In [38]:
pivot = scorecard_tn.groupby(['cip4','cred_desc']).size().unstack(fill_value=0)

valid_cips = pivot[
    (pivot["Bachelor's Degree"] > 0) &
    (pivot["Associate's Degree"] > 0) &
    (pivot["Undergraduate Certificate or Diploma"] > 0)
]

valid_cips.head(10)

cred_desc,Associate's Degree,Bachelor's Degree,Doctoral Degree,First Professional Degree,Graduate/Professional Certificate,Master's Degree,Post-baccalaureate Certificate,Undergraduate Certificate or Diploma
cip4,,,,,,,,
0100,1,4,1,0,0,1,0,1
0183,4,1,0,0,0,0,0,3
1002,3,5,0,0,0,2,0,5
1101,17,20,0,0,2,8,0,10
1102,1,1,0,0,0,0,0,1
1104,1,2,0,0,1,1,0,2
1108,7,7,0,0,1,1,0,6
1110,4,6,0,0,1,2,0,18
1205,6,1,0,0,1,0,0,8


In [40]:
valid_cips.shape

(52, 8)

In [42]:
# Verify that scorecard and completions cip4 are the same.
scorecard_tn['cip4'].head(10)

0    2401
1    2401
2    2401
3    3802
4    3906
5    3906
6    3999
7    3999
8    3999
9    4400
Name: cip4, dtype: object

In [43]:
scorecard_tn['cip4'].value_counts()

cip4
5202    134
5138    113
2401     85
5109     73
1312     68
       ... 
0405      1
0406      1
0105      1
1106      1
5126      1
Name: count, Length: 292, dtype: int64